# Sanity 06 — the recommendation subplot

Mirrors `scripts/next_merchant_probe.py` (+ `_v3`) on the research branch. Same frozen
embedding, next-merchant prediction over a 21k-way merchant token space, everything on
identical sampled test events in raw-merchant space. The story is the readout ladder, the
honest memorization floor, and the hybrid that finally clears it.

In [ ]:
import os, json
import numpy as np
import pandas as pd

BASE = os.environ.get("FM_BASE", "/mnt/user_storage/transaction-fm-v2")

In [ ]:
d = f"{BASE}/downstream/full_nextmerchant"
v2 = json.load(open(f"{d}/probe_metrics_v2.json"))
v3 = json.load(open(f"{d}/probe_metrics_v3.json"))
ladder = pd.DataFrame({
    "readout": ["masked-position (original 06 eval)", "InfoNCE table, zero-shot",
                "linear head", "2-layer MLP", "MLP + time-context feats (v3)"],
    "hr@10": [0.077,
              v2["readouts"]["infonce_zeroshot"]["hr@10"],
              v2["readouts"]["linear"]["hr@10"],
              v2["readouts"]["mlp"]["hr@10"],
              v3["readouts"]["mlp_solo_token_space"]["hr@10"]],
}).round(3)
ladder

In [ ]:
floor_and_hybrid = pd.DataFrame({
    "method": ["naive: causal full-history top-10 (the honest floor)",
               "naive + recency decay (tau=%dd)" % v3["readouts"]["naive_recency"]["tau_days"],
               "logistic candidate re-ranker",
               "HYBRID: 0.1*softmax(MLP) + 0.9*frequency prior"],
    "hr@10": [v3["readouts"]["naive_count"]["hr@10"],
              v3["readouts"]["naive_recency"]["hr@10"],
              v3["readouts"]["ranker"]["hr@10"],
              v3["readouts"]["alpha_blend"]["hr@10"]],
}).round(4)
floor_and_hybrid

In [ ]:
# Where the FM earns its budget: slices where memorization is structurally blind.
print(json.dumps(v3["slices"], indent=2))
print("ranker feature coefficients:", v3["readouts"]["ranker"]["coef"])

**What to check:** the hybrid (0.658) beats the honest floor (0.647), paired on 400k events.
The slices are the real story: 35% of test events have a next merchant outside the card's
top-10 — naive scores 0.000 there, the model recovers ~0.16; on never-seen merchants only the
full-vocab MLP scores at all (~0.077). In the ranker coefficients, note `inf_ls` ≈ 0.01: the
model's own InfoNCE merchant scorer (the "surprise" signal) adds nothing as a ranking feature —
that hypothesis was tested and closed. The trained MLP head is saved at
`model/full/next_merchant_mlp.pt` (input = z-scored [embedding ⊕ hour ⊕ dow ⊕ log-gap]).